<a href="https://colab.research.google.com/github/khine-thant-su/crisis_companion_chatbot/blob/main/safety_classifier_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description of notebook contents
This notebook uses **TfidfVectorizer** to vectorize text data, and uses the vectorized outputs to train a **Logistic Regression** classifier that returns a prediction of a given user input into one of three categories - "depression", "SuicideWatch", "teenagers" (categories represented in the data used to train the model).

**The final Logistic Regression model achieves an accuracy of 0.7675 on validation data (N=714), and 0.7647 on test data (N=1428).**

⭐ NOTES:

Dataset used for training the classifier: [Suicide Depression Detection](https://huggingface.co/datasets/joshyii/suicide_depression_detection)

Read more about how TfidfVectorizer works [here](https://codepointtech.com/mastering-text-data-tfidfvectorizer-in-scikit-learn/).

This safety classifier can be integrated with the chatbot as follows:

* User input -> Classifier -> Risk scores (Predicted probabilities of a response belonging to a class) -> Apply thresholds for follow-up action.



In [1]:
!pip install --upgrade nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [47]:
import pandas as pd
import pyarrow.parquet as pq  # To read the parquet file

import string  # To preprocess text data
import nltk
from nltk.stem import WordNetLemmatizer as wnl
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')  # For part-of-speech (POS) tagging
nltk.download('punkt_tab')  # For sentence tokenization

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
RANDOM_STATE = 42

In [4]:
# Read the Parquet file into an Arrow Table
table = pq.read_table('/content/0000.parquet')

# Convert the Arrow Table to a Pandas DataFrame
df = table.to_pandas()
print(df)

                                                     text         class
0       Does life actually work for most / non-depress...    depression
1       I found my friend's bodyIt was almost nine yea...    depression
2       Ex Wife Threatening SuicideRecently I left my ...  SuicideWatch
3       Am I weird I don't get affected by compliments...     teenagers
4       Finally 2020 is almost over... So I can never ...     teenagers
...                                                   ...           ...
348119  You how you can tell i have so many friends an...     teenagers
348120  pee probably tastes like salty tea😏💦‼️ can som...     teenagers
348121  The usual stuff you find hereI'm not posting t...  SuicideWatch
348122  I confronted my mother. Extremely isolated, wi...    depression
348123  I still haven't beaten the first boss in Hollo...     teenagers

[348124 rows x 2 columns]


In [5]:
# Check for missing values in either column
df.isna().sum()

,0
text,1
class,14


In [6]:
# Observations with missing class -- These could be used as test data maybe?
df.loc[df['class'].isna()]

,text,class
11557,I feel like im in a nightmare.Something happen...,None
11558,It's like I'm living in a nightmare and everyt...,None
11559,(view post history for more info on my dad),None
41048,A doodle of my struggle with depressionhttp://...,None
47570,Thinking of putting this as my profile picture...,None
61160,If I told you I want to move on with my life a...,None
141715,I think I might need someone to talk me down f...,None
141716,I've known that I'll never get any love outsid...,None
156657,A clip that describes how I feel when I'm tryi...,None
156658,depression,None


In [7]:
# Observation with missing text
df.loc[df['text'].isna()]

,text,class
185323,None,depression


In [8]:
# Check for very short texts that might be noise
print("Number of short text entries:", len(df.loc[df['text'].str.strip().str.len() < 10]), "\n")
df.loc[df['text'].str.strip().str.len() < 10].head()

Number of short text entries: 35 



,text,class
11019,okok,SuicideWatch
19013,f you :),teenagers
24908,Hello:],SuicideWatch
28983,Hello.:),SuicideWatch
29193,deadme?,SuicideWatch


In [9]:
# There is equal class distribution across the three classes.
df['class'].value_counts()

,count
class,
SuicideWatch,116037
teenagers,116037
depression,116036


In [10]:
# Check for duplicate entries in the 'text' column -- no duplicate prompts.
duplicate_entries = df[df['text'].duplicated(keep=False)]
display(duplicate_entries)

,text,class


In [11]:
df.sample(3, random_state=RANDOM_STATE)['text']

,text
182709,I never thought I would be cheated on. But her...
18177,should i switch from eclipse to visual studio ...
252758,Good in this evil worldI've been through pain ...


In [12]:
# Prepare subset dataset before train_test_split
# 1. Drop rows where class or text is None
df_subset = df[(df['class'].notna()) & (df['text'].notna())]

# 2. Drop rows where text is too short
df_subset = df_subset[df_subset['text'].str.strip().str.len() > 10]
len(df_subset)

348061

In [13]:
# Check for class balance in the dataset
df_subset['class'].value_counts()

,count
class,
teenagers,116028
depression,116026
SuicideWatch,116007


### Train-val-test split

In [14]:
# Train-val-test split
X = df_subset['text'].values
y = df_subset['class'].values

# 80% temp (train + val), 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)  # Ensures that the split is performed in a way that maintains the same proportion of classes in both the training and testing datasets as in the original dataset

# Split the 80% into 70% train, 10% val (10% of the original dataset will be saved for validation)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE, stratify=y_temp)  # 0.8 * 0.125 = 0.10


In [15]:
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 243642, Val: 34806, Test: 69613


In [16]:
# Preview the first few elements of X_train
# X_train is an array of strings.
display(X_train[120])

'Question about tooth extraction Does it hurt too much when the dentist removes a tooth that is not loose? Seriously, I do not know where to ask this, but I need a positive answer to relieve my stress.'

In [17]:
# Create subset arrays for training because otherwise RAM will run out.
X_train_subset = X_train[:5000]
y_train_subset = y_train[:5000]

Build pipeline (TF-IDF → Logistic Regression)

In [18]:
# Example for how nltk.pos_tag() works
# print(nltk.pos_tag(['feet']))
# print(nltk.pos_tag(['feet'])[0][1][0])

In [19]:
import re # Import the re module for regular expressions
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords # Import stopwords
import nltk
from nltk.stem import WordNetLemmatizer as wnl
import string

# Initialize WordNetLemmatizer
lemmatizer = wnl()

def get_wordnet_pos(word):
    """Map NLTK POS tags to WordNet POS tags"""
    tag = nltk.pos_tag([word])[0][1][0].upper()  # Extract the first letter
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN) # Default to noun if POS not found

# Compile a regex to find characters repeated 3 or more times consecutively
repeat_regex = re.compile(r"(.)\1{2,}") # . matches any character, \1 matches the same character as the first capturing group, {2,} matches 2 or more repetitions

def clean(text):
    '''Converts the input text into lower case, removes <br> tags, punctuation, whitespace, and stopwords. Lemmatizes the words using POS tags.
    Returns the processed words in a list.

        Args:
        text(str): input text'''

    text = "".join([i.lower() for i in text if i not in string.punctuation])  # Only keep non-punctuation characters. Each "i" is a letter, not a word. "".join() returns a sentence.
    words = word_tokenize(text)  # Tokenize the text into words
    words = [word for word in words if word not in stopwords.words('english')]  # Remove stopwords
    words = [word for word in words if word.isalpha()]  # Keep only alphabetic characters
    words = [word for word in words if len(word) >= 2 and len(word) <= 20]  # Remove words shorter than 2 or longer than 20 characters
    words = [word for word in words if not repeat_regex.search(word)]  # Remove words with excessive character repetition
    text = ' '.join([lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words])  # Lemmatize words based on their POS tags

    return text

### Preprocess text before feeding it into TfidfVectorizer

In [20]:
# This code will take a while to run.
X_train_subset_cleaned = [clean(text) for text in X_train_subset]

In [21]:
# 1. Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))  # Extract both unigrams and bigrams

# 2. Fit and transform the training data
# fit() learns the vocabulary and IDF values
# transform() converts the text to TF-IDF features
tfidf_matrix = tfidf_vectorizer.fit_transform(X_train_subset_cleaned)  # tfidf_matrix will be a sparse matrix

# Convert to a dense array for easier viewing
print(tfidf_matrix.toarray()[:10])

# Get the feature names. A feature is a unique word or a sequence of words (n-grams) in the vocabulary that TfidfVectorizer has built, based on the documents it's been fitted on.
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nFeature Names (Vocabulary):\n")
print(feature_names[:100])

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Feature Names (Vocabulary):

['aa' 'aa aa' 'aa afraid' 'aa alcoholism' 'aa degree' 'aa get'
 'aa meeting' 'aand' 'aand hahah' 'ab' 'ab exist' 'ab muscle' 'ab watch'
 'abandon' 'abandon also' 'abandon apartment' 'abandon best'
 'abandon brother' 'abandon career' 'abandon child' 'abandon despite'
 'abandon ecuador' 'abandon even' 'abandon ever' 'abandon filler'
 'abandon furry' 'abandon glory' 'abandon go' 'abandon guess'
 'abandon hell' 'abandon hope' 'abandon make' 'abandon many'
 'abandon mesorry' 'abandon miserable' 'abandon parent'
 'abandon psychologist' 'abandon ready' 'abandon say' 'abandon sometimes'
 'abandon still' 'abandon take' 'abandon time' 'abandon toxic'
 'abandon treatment' 'abandon twice' 'abandon two' 'abandon well'
 'abandonment' 'abandonment depression' 'abandonment dont'
 'abandonment get' 'abandonment issue' '

In [22]:
# Each text is a document (num of rows). Num of cols shows the unique features (unigrams and bigrams) across the documents.
tfidf_dense_array = tfidf_matrix.toarray()
tfidf_dense_array.shape

(5000, 239944)

In [23]:
# # Check why there are numbers in the text data
# for text in X_train_subset_cleaned:
#   if '003478628' in text:
#     display(text)

### We're ready to model!

[Optional] We can also try grid search to find the best parameters for TfidfVectorizer and Logistic Regression classifier.

In [24]:
def train_lr_model(tfidf_arrays, labels):
    '''Trains and returns a Logistic Regression model.

        Args:
        tfidf_arrays(array): arrays created by transforming text into a TF-IDF document-term matrix
        labels(array): sentiment labels in the form of an array'''

    classifier = LogisticRegression().fit(tfidf_arrays, labels)

    return classifier

In [26]:
# Train the Logistic Regression model
lr_model = train_lr_model(tfidf_matrix, y_train_subset)

In [36]:
# Create smaller subsets for val and test that keep the same proportion of train-val-test split (70%-10%-20%)
X_val_subset = X_val[:714]  # 10% of 7142
X_test_subset = X_test[:1428]  # 20% of 7142

y_val_subset = y_val[:714]  # 10% of 7142
y_test_subset = y_test[:1428]  # 20% of 7142

# Clean the val and test text inputs (WITHOUT refitting the vectorizer on new, unseen data)
X_val_subset_cleaned = [clean(text) for text in X_val_subset]
X_test_subset_cleaned = [clean(text) for text in X_test_subset]

In [37]:
# Make smaller subsets for val and test that keep the same proportion of train-val-test split (70%-10%-20%)
y_val_subset = y_val[:714]  # 10% of 7142
y_test_subset = y_test[:1428]  # 20% of 7142

❗NOTE:

Once you have trained your TfidfVectorizer on your training data, it is crucial to use the same fitted vectorizer to transform any new, unseen data (e.g., test sets or new incoming text).

**You should only call transform() on new data, not fit_transform().**

In [38]:
# Transform the raw validation text to TFIDF features using the same fitted vectorizer from before
val_tfidf_matrix = tfidf_vectorizer.transform(X_val_subset_cleaned)

In [43]:
lr_model_accuracy = lr_model.score(val_tfidf_matrix, y_val_subset)
print(f"LR model accuracy: {lr_model_accuracy:.4f}")  # Proportion of validation samples correctly classified by my trained lr_model

LR model accuracy: 0.7591


### Display the predictions alongside the true labels for the first few samples

In [45]:
# Get predictions for the validation set
y_pred_val = lr_model.predict(val_tfidf_matrix)

from sklearn.metrics import accuracy_score
print("Accuracy score on val data:", round(accuracy_score(y_val_subset, y_pred_val), 4))
print("______________________")

print("Predictions vs True Labels (first 3 samples):")
for i in range(3):
    print(f"User message: {X_val_subset[i]}, Predicted: {y_pred_val[i]}, True: {y_val_subset[i]}")
    print("______________________")

Accuracy score on val data: 0.7591
______________________
Predictions vs True Labels (first 3 samples):
User message: How is someone a homophobe? Like they’re literally a HOMO Sapien. Dumbass people these days smh. 🤦‍♂️, Predicted: teenagers, True: teenagers
______________________
User message: Fuck my life, fuck everything.Alright it took me a long time to post here. I feel honestly like I'm just asking for attention and this is in some way the wrong way to do this. I've tried talking to my best friend but he just stopped replying.

So everything was finally looking good, I'm in sophomore year and I was making new fiends, going to parties, and just having a great time with family and friends.

After Christmas shit hit the fan. My grandpa, who is my main father figure since my dad isn't around much, got a new girlfriend after my grandma died. She was like my mom, I was closed with her then my mom, and she died and I had to watch it, literally I watched her die in the most painful way i

### Try parameter tuning with grid search

First, check for class imbalance in the train_subset data. This would dictate which scoring method we should use for grid search results.

If we don't have a high class imbalance, we can proceed with using "accuracy" as a measure.

In [51]:
# Convert y_train to a pandas Series and get value counts
train_subset_class_counts = pd.Series(y_train_subset).value_counts()

print("Class distribution in X_train_subset (based on y_train_subset):")
print(train_subset_class_counts)

Class distribution in X_train_subset (based on y_train_subset):
depression      1679
SuicideWatch    1676
teenagers       1645
Name: count, dtype: int64


In [58]:
# Pipeline
pipe = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1,1),        # will be tuned
        min_df=2,                 # will be tuned
        max_df=0.95               # will be tuned
    )),
    ("clf", LogisticRegression(
        max_iter=2000,  # Sets the maximum number of iterations for the solver to converge
        solver="lbfgs",  # Specifies the algorithm to use for the optimization problem
        multi_class="auto",
        class_weight="balanced",
        random_state=RANDOM_STATE  # Seeds the random number generator used by the solver (if the solver uses randomness), so running the code again with the same random_state will result in the same model being trained
    ))
])


# We define the grid as a dictionary where keys are the names of the parameters and the values are the list of values to test for each parameter.
param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [2, 5],  # Filters terms based on their document frequency. min_df ignores terms that appear in too few documents
    "tfidf__max_df": [0.9, 0.95],  #  max_df ignores terms that appear in too many (e.g., above a certain percentage)
    "clf__C": [0.5, 1.0, 2.0, 4.0]  # larger C = less regularization
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # StratifiedKFold is a variation of K-Fold cross-validation that ensures each fold has the same proportion of samples of each target class as the whole dataset.


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_subset_cleaned, y_train_subset)  # This tells TfidfVectorizer to do fit_transform().

print("Best CV params:", grid.best_params_)
print("Best CV score (accuracy):", grid.best_score_)


Fitting 5 folds for each of 32 candidates, totalling 160 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Best CV params: {'clf__C': 4.0, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Best CV score (accuracy): 0.7659999999999999


In [59]:
# [Optional] If we are curious how the new parameters changed the TfidfVectorized outputs:
# 1. Retrain TfidfVectorizer with parameters suggested by grid search
tfidf_vectorizer_retrained = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_df=0.9, min_df=2)

tfidf_matrix_retrained = tfidf_vectorizer_retrained.fit_transform(X_train_subset_cleaned)

feature_names = tfidf_vectorizer_retrained.get_feature_names_out()
print("\nFeature Names (Vocabulary):\n")
print(feature_names[:100])


Feature Names (Vocabulary):

['aa' 'aa get' 'ab' 'abandon' 'abandon still' 'abandonment'
 'abandonment issue' 'abd' 'abhorrent' 'ability' 'ability even'
 'ability feel' 'ability focus' 'ability help' 'ability put' 'abject'
 'abject misery' 'able' 'able actually' 'able admit' 'able afford'
 'able anything' 'able break' 'able bring' 'able buy' 'able call'
 'able change' 'able come' 'able communicate' 'able control' 'able cope'
 'able deal' 'able eat' 'able enjoy' 'able even' 'able experience'
 'able feel' 'able fight' 'able find' 'able finish' 'able focus'
 'able fuck' 'able function' 'able get' 'able give' 'able go'
 'able handle' 'able happy' 'able help' 'able hide' 'able hold' 'able job'
 'able keep' 'able kill' 'able leave' 'able let' 'able lie' 'able listen'
 'able live' 'able love' 'able make' 'able move' 'able open' 'able pay'
 'able play' 'able provide' 'able pull' 'able put' 'able relate'
 'able return' 'able say' 'able see' 'able shake' 'able sit' 'able sleep'
 'able smoke' 'a

In [60]:
# The best model suggested by grid search
best_model = grid.best_estimator_
best_model

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2),
                                 strip_accents='unicode')),
                ('clf',
                 LogisticRegression(C=4.0, class_weight='balanced',
                                    max_iter=2000, multi_class='auto',
                                    random_state=42))])

In [61]:
# Validate on the val set
from sklearn.metrics import classification_report
y_val_pred = best_model.predict(X_val_subset_cleaned)
print("\nValidation classification report:")
print(classification_report(y_val_subset, y_val_pred, digits=4))



Validation classification report:
              precision    recall  f1-score   support

SuicideWatch     0.7617    0.6792    0.7181       240
  depression     0.6780    0.7207    0.6987       222
   teenagers     0.8523    0.8929    0.8721       252

    accuracy                         0.7675       714
   macro avg     0.7640    0.7642    0.7629       714
weighted avg     0.7676    0.7675    0.7664       714



In [63]:
print("Accuracy score on val data (after grid search):", round(accuracy_score(y_val_subset, y_val_pred), 4))

Accuracy score on val data (after grid search): 0.7675


In [64]:
y_test_pred = best_model.predict(X_test_subset_cleaned)
print("\nTest set classification report:")
print(classification_report(y_test_subset, y_test_pred, digits=4))


Test set classification report:
              precision    recall  f1-score   support

SuicideWatch     0.7309    0.6731    0.7008       468
  depression     0.6852    0.7191    0.7018       445
   teenagers     0.8623    0.8874    0.8746       515

    accuracy                         0.7647      1428
   macro avg     0.7594    0.7599    0.7591      1428
weighted avg     0.7640    0.7647    0.7638      1428



In [65]:
print("Accuracy score on test data (after grid search):", round(accuracy_score(y_test_subset, y_test_pred), 4))

Accuracy score on test data (after grid search): 0.7647
